In [8]:
# Import required libraries
import os
import pandas as pd

In [9]:
# Load the dataset and select relevant columns
DATA_PATH = "../../data"
CLEAN_DATA_PATH = "../../cleaned_data"

df = pd.read_csv(DATA_PATH + '/train_participants.tsv', sep='\t')
df

,participant_id,subject,biological_sex,race,min_age,max_age,geolocation,investigation_id,investigation_name,arm_id,arm_name,data_source,description,basic_curation,pubmed_ids,main_pmid,main_publication_author
0,SDY269.SUB112836,SUB112836,female,White,28,28,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
1,SDY269.SUB112849,SUB112849,female,Black or African American,39,39,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
2,SDY269.SUB112854,SUB112854,male,Black or African American,46,46,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
3,SDY269.SUB112860,SUB112860,female,White,32,32,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
4,SDY269.SUB112881,SUB112881,female,Black or African American,29,29,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3955,EXT101.Y2,Y2,female,White,28,28,US: Connecticut,EXT101,High-throughput single-cell profiling of B cel...,Shaw_2012_2013,2012_2013_Seasonal_vaccine,External,High-throughput single-cell/bulk profiling of ...,no,37367734,37367734,Wang M (2023)
3956,EXT101.Y3,Y3,male,White,29,29,US: Connecticut,EXT101,High-throughput single-cell profiling of B cel...,Shaw_2014_2015,2014_2015_Seasonal_vaccine,External,High-throughput single-cell/bulk profiling of ...,no,37367734,37367734,Wang M (2023)
3957,EXT101.O1,O1,female,White,66,66,US: Connecticut,EXT101,High-throughput single-cell profiling of B cel...,Shaw_2014_2015,2014_2015_Seasonal_vaccine,External,High-throughput single-cell/bulk profiling of ...,no,37367734,37367734,Wang M (2023)
3958,EXT101.O2,O2,male,White,85,85,US: Connecticut,EXT101,High-throughput single-cell profiling of B cel...,Shaw_2012_2013,2012_2013_Seasonal_vaccine,External,High-throughput single-cell/bulk profiling of ...,no,37367734,37367734,Wang M (2023)


In [10]:
# Race and geolocation only have 1 value in the challenge set
df = df[[
    'participant_id',
    'biological_sex',
    # 'race',
    'min_age',
    'max_age',
    # 'geolocation',
    'arm_name'
]]
df.head()

,participant_id,biological_sex,min_age,max_age,arm_name
0,SDY269.SUB112836,female,28,28,LAIV group 2008
1,SDY269.SUB112849,female,39,39,LAIV group 2008
2,SDY269.SUB112854,male,46,46,LAIV group 2008
3,SDY269.SUB112860,female,32,32,LAIV group 2008
4,SDY269.SUB112881,female,29,29,LAIV group 2008


In [11]:
# Feature engineering: Calculate average age and drop original min/max age columns
df['age'] = (df['min_age'] + df['max_age']) / 2
df = df.drop(columns=['min_age', 'max_age'])

# Clean biological_sex column (Standardize Male/Female)
df['biological_sex'] = df['biological_sex'].str.lower()

# Simplify arm_name to vaccine dose category
def simplify_arm(arm):
    if isinstance(arm, str):
        if 'High Dose' in arm:
            return 'High Dose Fluzone'
        elif 'Standard' in arm or 'standard' in arm:
            return 'Standard Fluzone'
    return 'Other'

df['arm_name'] = df['arm_name'].apply(simplify_arm)

df.head()

,participant_id,biological_sex,arm_name,age
0,SDY269.SUB112836,female,Other,28.0
1,SDY269.SUB112849,female,Other,39.0
2,SDY269.SUB112854,male,Other,46.0
3,SDY269.SUB112860,female,Other,32.0
4,SDY269.SUB112881,female,Other,29.0


In [12]:
# Print unique values for each column to verify cleaning
for col in df.columns:
    print(f"{col}: {df[col].unique()}\n")

participant_id: <ArrowStringArray>
[ 'SDY269.SUB112836',  'SDY269.SUB112849',  'SDY269.SUB112854',
  'SDY269.SUB112860',  'SDY269.SUB112881',  'SDY269.SUB112886',
  'SDY269.SUB112846',  'SDY269.SUB112857',  'SDY269.SUB112872',
  'SDY270.SUB112872',
 ...
 'SDY2867.SUB389726',     'EXT100.321P04',     'EXT100.321P05',
     'EXT100.321P11',         'EXT101.Y1',         'EXT101.Y2',
         'EXT101.Y3',         'EXT101.O1',         'EXT101.O2',
         'EXT101.O3']
Length: 3960, dtype: str

biological_sex: <ArrowStringArray>
['female', 'male']
Length: 2, dtype: str

arm_name: <ArrowStringArray>
['Other', 'Standard Fluzone', 'High Dose Fluzone']
Length: 3, dtype: str

age: [28.  39.  46.  32.  29.  43.  31.  33.  25.  35.  44.  24.  65.  69.
 86.  90.  27.  76.  64.  68.  85.  89.  30.  67.  71.  88.  22.  26.
 72.  66.  70.  75.  78.  82.  61.  51.  62.  49.  74.  50.  54.  45.
 23.  87.  84.  80.  57.  37.  48.  53.  36.  56.  20.  21.  19.  79.
 40.  77.  55.  58.  73.  63.  59.  81.  

In [13]:
df.columns = ['participant_id'] + [f'PART_{c}' for c in df.columns[1:]]
df.head()

,participant_id,PART_biological_sex,PART_arm_name,PART_age
0,SDY269.SUB112836,female,Other,28.0
1,SDY269.SUB112849,female,Other,39.0
2,SDY269.SUB112854,male,Other,46.0
3,SDY269.SUB112860,female,Other,32.0
4,SDY269.SUB112881,female,Other,29.0


In [14]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs(CLEAN_DATA_PATH, exist_ok=True)
df.to_csv(CLEAN_DATA_PATH + '/participants_cleaned.csv', index=False)